# Model 4: XGBoost for Supervised Fraud Detection
This notebook implements a supervised Extreme Gradient Boosting (XGBoost) algorithm as a primary classification layer to identify complex and non-linear patterns in fraudulent financial transactions.Unlike the unsupervised approach of Isolation Forest, XGBoost learns from labeled historical data to provide high-precision detection. 

The model is trained using the processed IEEE-CIS Fraud Detection dataset and rigorously evaluated across multiple environments (ULB, IEEE-CIS, and PaySim datasets) using standard classification metrics such as:

- Precision 
- Recall 
- F1-score 
- Confusion Matrix 
- Feature Importance (Advanced HD-level analysis) 

In [1]:
import os
import pandas as pd
import numpy as np
import joblib
import json
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
def load_data(file_name):
    paths_to_try = [
        f'../../data/processed/{file_name}',  
        f'data/processed/{file_name}',        
        f'../data/processed/{file_name}'      
    ]
    
    for p in paths_to_try:
        if os.path.exists(p):
            print(f"✅ Found and successfully loaded: {p}")
            return pd.read_csv(p)
            
    print(f"❌ Could not find {file_name}!")
    print(f"Your Jupyter environment is currently running from: {os.getcwd()}")
    return None

print("Searching for and loading datasets...")
df_ulb = load_data('processed_ulb_dataset.csv')
df_ieee = load_data('processed_ieee_dataset.csv')
df_paysim = load_data('processed_paysim_dataset.csv')

In [ ]:
print("--- Training XGBoost on ULB Dataset ---")
X_u = df_ulb.drop('Class', axis=1)
y_u = df_ulb['Class']

X_train_u, X_test_u, y_train_u, y_test_u = train_test_split(X_u, y_u, test_size=0.2, random_state=42)

xgb_ulb = XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, eval_metric='logloss')
xgb_ulb.fit(X_train_u, y_train_u)

y_pred_u = xgb_ulb.predict(X_test_u)

print("\n=== EVALUATION: ULB DATASET ===")
print(classification_report(y_test_u, y_pred_u))

plt.figure(figsize=(6,4))
sns.heatmap(confusion_matrix(y_test_u, y_pred_u), annot=True, fmt='d', cmap='Blues')
plt.title('XGBoost - ULB Confusion Matrix')
plt.show()

In [ ]:
print("--- Training XGBoost on IEEE Dataset ---")
X_i = df_ieee.drop('isFraud', axis=1)
y_i = df_ieee['isFraud']

X_train_i, X_test_i, y_train_i, y_test_i = train_test_split(X_i, y_i, test_size=0.2, random_state=42)

xgb_ieee = XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, eval_metric='logloss')
xgb_ieee.fit(X_train_i, y_train_i)

y_pred_i = xgb_ieee.predict(X_test_i)

print("\n=== EVALUATION: IEEE DATASET ===")
print(classification_report(y_test_i, y_pred_i))

plt.figure(figsize=(6,4))
sns.heatmap(confusion_matrix(y_test_i, y_pred_i), annot=True, fmt='d', cmap='Greens')
plt.title('XGBoost - IEEE Confusion Matrix')
plt.show()

In [ ]:
print("--- Training XGBoost on PaySim Dataset ---")
X_p = df_paysim.drop('isFraud', axis=1)
y_p = df_paysim['isFraud']

X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(X_p, y_p, test_size=0.2, random_state=42)

xgb_paysim = XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, eval_metric='logloss')
xgb_paysim.fit(X_train_p, y_train_p)

y_pred_p = xgb_paysim.predict(X_test_p)

print("\n=== EVALUATION: PAYSIM DATASET ===")
print(classification_report(y_test_p, y_pred_p))

plt.figure(figsize=(6,4))
sns.heatmap(confusion_matrix(y_test_p, y_pred_p), annot=True, fmt='d', cmap='Purples')
plt.title('XGBoost - PaySim Confusion Matrix')
plt.show()

In [ ]:
plt.figure(figsize=(10,6))
feat_importances = pd.Series(xgb_ieee.feature_importances_, index=X_i.columns)
feat_importances.nlargest(10).plot(kind='barh', color='steelblue')
plt.title('Top 10 Important Features (XGBoost - IEEE)')
plt.xlabel('F-Score')
plt.show()

In [ ]:
cm = confusion_matrix(y_test_i, y_pred_i)
metrics_ieee = {
    "accuracy": round(float(accuracy_score(y_test_i, y_pred_i)), 4),
    "precision": round(float(precision_score(y_test_i, y_pred_i, zero_division=0)), 4),
    "recall": round(float(recall_score(y_test_i, y_pred_i, zero_division=0)), 4),
    "f1_score": round(float(f1_score(y_test_i, y_pred_i, zero_division=0)), 4),
    "confusion_matrix": {
        "true_negatives": int(cm[0,0]),
        "false_positives": int(cm[0,1]),
        "false_negatives": int(cm[1,0]),
        "true_positives": int(cm[1,1])
    }
}

model4_json = {
  "model_info": {
    "model_name": "XGBoost Classifier",
    "model_type": "Supervised Learning",
    "algorithm": "Extreme Gradient Boosting",
    "developer": "Ho Cao Cuong - 105687343"
  },
  "hyperparameters": {
    "n_estimators": 100,
    "max_depth": 6,
    "learning_rate": 0.1,
    "random_state": 42
  },
  "datasets_used": [
    "ULB Credit Card Fraud",
    "IEEE-CIS Fraud Detection",
    "PaySim Mobile Money"
  ],
  "performance_metrics": {
    "IEEE_Dataset": metrics_ieee
  },
  "saved_artifacts": {
    "model_file": "xgboost_ieee.pkl",
    "notebook_file": "Model4_Xgboost.ipynb"
  }
}

with open("model4_info.json", "w", encoding="utf-8") as f:
    json.dump(model4_json, f, indent=4)


joblib.dump(xgb_ieee, "xgboost_ieee.pkl")

print("✅ Successfully created 'model4_info.json' and 'xgboost_ieee.pkl'!")